# <b>ПЛАН Б v7: дообучение Айки (RP + вайфу) в Google Colab</b>

Датасет: смешанный <code>aika_train.jsonl</code> — русский ролеплей + аниме-архетипы (вайфу). Unsloth: в 2 раза быстрее старого transformers-варианта.

<b>Порядок:</b> Среда выполнения → Сменить среду выполнения → <b>T4 GPU</b>

In [ ]:
# 1. Проверка GPU. Должно быть Tesla T4 и ~15 ГБ
!nvidia-smi

In [ ]:
# 2. Ставим Unsloth (~2-3 мин)
%%capture
!pip install unsloth

In [ ]:
# 3. Подключаем Google Диск (появится ссылка → войди → разреши)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Забираем смешанный датасет RP + waifu из репы aika-finetune
!wget -q https://github.com/lucifermornngstar52-cell/aika-finetune/raw/main/data/aika_train.jsonl -O aika_train.jsonl


import json
lines = open('aika_train.jsonl', encoding='utf-8').read().strip().split('\n')
from datasets import Dataset
raw = [json.loads(l) for l in lines]
dataset = Dataset.from_list(raw)
waifu = sum(1 for r in raw if 'вайфу' in r['messages'][0]['content'][:200])
print(f'всего: {len(raw):,} примеров | вайфу: {waifu:,} | ролеплей: {len(raw)-waifu:,}')

In [ ]:
# 5. Загружаем мозг Qwen2.5-1.5B-Instruct в 4 битах + LoRA
from unsloth import FastLanguageModel

model, tok = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    max_seq_length=2048, dtype=None, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42)

In [ ]:
# 6. Разворачиваем messages в текст чата Qwen
def to_text(ex):
    return {'text': [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in ex['messages']]}

dataset = dataset.map(to_text, batched=True, remove_columns=dataset.column_names)
print(dataset[0]['text'][:300])

In [ ]:
# 7. Обучение (~40-60 мин на T4). Чекпоинт на Диск каждые 500 шагов.
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        output_dir='/content/drive/MyDrive/aika-checkpoints',
        num_train_epochs=3,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-4, warmup_ratio=0.03, weight_decay=0.01,
        logging_steps=25, save_strategy='steps', save_steps=500, save_total_limit=1,
        save_only_model=True, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        optim='adamw_8bit', lr_scheduler_type='cosine', seed=42, report_to='none'))

trainer.train()
print('ГОТОВО. Последний loss:', trainer.state.log_history[-1].get('train_loss', '?'))

In [ ]:
# 8. Проверяем, как она теперь говорит
FastLanguageModel.for_inference(model)

msgs = [{'role':'system','content':'Ты — Аика, аниме-девушка (вайфу) с характером цундере. Отвечай ярко, в этом стиле.'},
        {'role':'user','content':'Привет! Как ты?'}]
ids = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.7, top_p=0.9)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# 9. Сохраняем LoRA на Диск (работает после отключения сессии)
model.save_pretrained('/content/drive/MyDrive/aika-lora-rp')
tok.save_pretrained('/content/drive/MyDrive/aika-lora-rp')
print('LoRA на Диске: MyDrive/aika-lora-rp')

In [ ]:
# 10. (опционально) Слияние в 16-bit — на Диск, ~3 ГБ
model.save_pretrained_merged('/content/drive/MyDrive/aika-merged-rp', tok)

In [ ]:
# 11. (опционально) GGUF для телефона / llama.cpp. ~5 мин, БЕЗ компиляции.
model.save_pretrained_gguf('/content/drive/MyDrive/aika-gguf-rp', tok, quantization_method='q4_k_m')
print('GGUF на Диске: MyDrive/aika-gguf-rp (файл *.Q4_K_M.gguf)')